# TP · Consumo eléctrico en el NEA

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lautaromgo/estadistica-aplicada-unne/blob/main/notebooks/tp/tp_consumo_electrico.ipynb)

**Trabajo práctico de la Unidad III · puntos 1 a 4 · Diplomatura en Data Analytics (UNNE)**

> ### ¿En qué localidad del NEA se consume más electricidad, y por qué?

Una distribuidora juntó las lecturas de 2025 de los medidores de 800 hogares de Corrientes,
Resistencia, Posadas, Formosa e Isla del Cerrito. Los datos son **inventados**, pero traen adentro los
problemas de cualquier base real: encontrarlos es parte del TP.

**Cómo funciona.**

- Las **cinco fórmulas clásicas** (media, varianza, desvío, coeficiente de variación y correlación)
  las escribís **a mano**, en Python puro. Abajo de cada una, una celda 🔒 la compara con pandas y te
  dice ✅ o ❌.
- **Todo lo demás se hace con pandas, numpy, matplotlib y seaborn.** La mayoría de esas celdas ya
  vienen hechas: las corrés y leés el resultado. Donde hay que decidir algo, completás los huecos `___`.
- Hay **seis preguntas ✍️**, que se contestan con lo que te muestra la notebook. Cada una trae una
  💡 **pista** de qué mirar.
- El cierre es un **dashboard de 2 × 2** que armás vos, con el hallazgo en tres líneas.

Mientras no completes una función, las celdas que la usan muestran `NotImplementedError`: es
esperable, no rompiste nada. Las celdas 🔒 no se modifican.

**En Colab:** antes de empezar, *Archivo → Guardar una copia en Drive*.

**Qué se entrega:** esta notebook ejecutada entera (*Entorno de ejecución → Reiniciar sesión y
ejecutar todo*), descargada con *Archivo → Descargar → Descargar .ipynb*. **Es el único archivo que se
entrega**: no hay que entregar imágenes, capturas ni el `.csv`.

| se evalúa | puntos |
|---|---|
| las cinco funciones (✅ y sin pandas ni numpy adentro) | 25 |
| la limpieza | 10 |
| las preguntas P1 a P6 | 45 |
| el dashboard y el hallazgo | 20 |

> **Sobre las preguntas.** Cada respuesta tiene que apoyarse en un número o un gráfico **de esta
> notebook**, y citarlo. Las respuestas no pueden estar generadas con IA.

## 0 · Setup

Corré esta celda y seguí. Carga las librerías y los datos: busca el archivo al lado de la notebook;
si no está, lo baja de internet; y si tampoco puede, te pide que lo subas (te lo pasa la cátedra).

In [ ]:
# 🔒 Setup. No modifiques esta celda.
import inspect
import io
import math
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

# Los colores del TP: cada localidad tiene siempre el mismo color.
AZUL, ROJO, GRIS = "#1E579B", "#D1495B", "#9AA5B1"
COLORES = {"Formosa": AZUL, "Posadas": "#E8A33D", "Resistencia": "#7B5EA7",
           "Corrientes": "#1AA0B8", "Isla del Cerrito": ROJO}

# ── Los datos: de una carpeta, de internet o subidos a mano ──────────────────
ARCHIVO = "consumo_electrico_nea.csv"
URL = ("https://raw.githubusercontent.com/Lautaromgo/estadistica-aplicada-unne/"
       "main/data/raw/" + ARCHIVO)
RUTAS = [ARCHIVO, "../../data/raw/" + ARCHIVO, "../data/raw/" + ARCHIVO, "data/raw/" + ARCHIVO]


def _leer_archivo():
    for ruta in RUTAS:                     # 1. ¿Está en alguna de estas carpetas?
        if os.path.exists(ruta):
            with open(ruta, encoding="utf-8") as f:
                return f.read(), ruta
    try:                                   # 2. Si no, lo baja de internet.
        with urllib.request.urlopen(URL, timeout=30) as respuesta:
            return respuesta.read().decode("utf-8"), "GitHub"
    except Exception:
        pass
    try:                                   # 3. Si tampoco puede y estás en Colab, te lo pide.
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"No encontré {ARCHIVO}: ponelo en la misma carpeta que esta notebook.")
    print(f"No pude bajar {ARCHIVO}. Subilo con el botón de abajo (te lo pasa la cátedra).")
    subido = files.upload()
    return next(iter(subido.values())).decode("utf-8"), "subido a mano"


texto, origen = _leer_archivo()
df = pd.read_csv(io.StringIO(texto))          # el archivo, tal como viene


# ── Funciones de ayuda, ya hechas ───────────────────────────────────────────
def nro(x, decimales=0):
    """Escribe un número con punto de miles y coma decimal:  nro(9680) -> "9.680"."""
    return f"{x:,.{decimales}f}".replace(",", "|").replace(".", ",").replace("|", ".")


def verificar(nombre, obtenido, esperado, tol=1e-6):
    """Compara lo que obtuviste con lo que se esperaba, y muestra ✅ o ❌.

    Los números se comparan con un margen de una millonésima, y si no coinciden
    muestra los dos valores. No corta la notebook.
    """
    try:
        ok = abs(float(obtenido) - float(esperado)) <= tol * max(1.0, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = obtenido == esperado
    print(f"✅ {nombre}" if ok
          else f"❌ {nombre} → obtuviste {obtenido!r}, se esperaba {esperado!r}")


___ = None          # los huecos que vas a completar

print(f"{nro(len(df))} filas × {df.shape[1]} columnas   |   origen: {origen}")

### Cómo se leen las verificaciones

Después de cada función que escribís hay una celda 🔒 que la pone a prueba. Todas hacen lo mismo: le
piden un resultado a **tu** función, le piden el mismo resultado a **pandas** (que acá sirve de
referencia), y comparan los dos con `verificar()`. Cada renglón tiene esta forma:

```
verificar("media del consumo", media(v), s.mean())
          └────────┬────────┘  └──┬───┘  └──┬───┘
             qué se prueba     lo tuyo    pandas
```

- **✅** quiere decir que los dos resultados coinciden.
- **❌** muestra lo que obtuviste y lo que se esperaba. Con esos dos números, volvé al `TODO` de la
  función.
- Los decimales se comparan con un margen mínimo (una millonésima), porque la misma cuenta hecha en
  otro orden puede diferir en el último decimal.

### Las diez columnas

| columna | qué dice |
|---|---|
| `id_hogar` | el código del hogar (el número de suministro) |
| `localidad` | Corrientes, Resistencia, Posadas, Formosa o Isla del Cerrito |
| `periodo` | el mes de la lectura, de `2025-01` a `2025-12` |
| `personas` | cuántas personas viven en el hogar |
| `superficie_m2` | la superficie de la vivienda, en m² |
| `aires_acondicionados` | cuántos equipos de aire acondicionado tiene |
| `nivel_subsidio` | el nivel en la segmentación de subsidios: **N1** ingresos altos, **N2** ingresos bajos, **N3** ingresos medios |
| `temperatura_media` | la temperatura media de ese mes en esa localidad, en °C |
| `consumo_kwh` | **la protagonista**: la energía que marcó el medidor ese mes, en kWh |
| `importe` | lo que se facturó ese mes, en pesos |

Las primeras filas, **tal como vienen** en el archivo:

In [ ]:
df.head(8)

---
## Parte 1 · ¿Puedo confiar en estos datos?  *(punto 1)*

Antes de calcular nada, hay que mirar qué hay en cada columna. `.info()` muestra cuántos datos tiene
cada una y de qué tipo los leyó pandas: `int64` y `float64` son números; `object` es **texto**.

In [ ]:
df.info()

Dos columnas que deberían ser números llegaron como texto: `consumo_kwh` e `importe`. Así vienen
escritos los números en este archivo:

| columna | ejemplo | qué tiene de particular |
|---|---|---|
| `temperatura_media` | `27.3` | punto decimal: pandas la leyó bien |
| `importe` | `$ 58.483,50` | signo pesos, punto de miles y **coma** decimal |
| `consumo_kwh` | `419` | un entero… casi siempre |

La celda de abajo los convierte en números con una función que ya viene hecha, `a_numero()`, y guarda
el resultado en dos columnas nuevas: `consumo` e `importe_pesos`.

In [ ]:
# 🔒 Texto -> números. No modifiques esta celda.
def a_numero(texto):
    """Convierte el texto de una celda en un número, o en NaN si ahí no hay un número.

        "419"           ->  419.0
        "$ 58.483,50"   ->  58483.5    (punto de miles y coma decimal)
        "s/l", "", "-"  ->  NaN        (así se escribe "no hay dato" en este archivo)
    """
    if pd.isna(texto):                          # una celda vacía ya llega como NaN
        return np.nan
    s = str(texto).replace("$", "").strip()
    if s in ("", "s/l", "-"):
        return np.nan
    if "," in s:                                # con coma decimal, los puntos son de miles
        s = s.replace(".", "").replace(",", ".")
    return float(s)


df["consumo"] = df["consumo_kwh"].map(a_numero)
df["importe_pesos"] = df["importe"].map(a_numero)
df[["consumo_kwh", "consumo", "importe", "importe_pesos"]].head()

### El perfil de cada columna

Para cada columna: cuántos datos faltan, cuántos valores distintos tiene, cuál es el que más se
repite y cuántas veces aparece. Ya está hecha: leela con atención, porque de acá salen las preguntas
que siguen. La celda que viene después cuenta las filas, los hogares y las filas **duplicadas**
(idénticas a otra): eso lo vas a usar en la limpieza.

In [ ]:
perfil = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "valores distintos": df.nunique(),
    "el más repetido": df.apply(lambda columna: columna.value_counts().index[0]),
    "cuántas veces": df.apply(lambda columna: columna.value_counts().iloc[0]),
})
perfil

In [ ]:
print(f"filas: {nro(len(df))}   ·   hogares distintos: {df['id_hogar'].nunique()}   ·   "
      f"filas duplicadas: {df.duplicated().sum()}")
print("\nDe dónde son las filas duplicadas:")
df[df.duplicated()][["localidad", "periodo"]].value_counts()

### ✍️ P1 · El valor que más se repite

En el perfil, mirá el valor más repetido de `consumo` y cuántas veces aparece. En una variable
continua, un valor exacto que se repite tanto casi nunca es casualidad.

- ¿Qué valor es, y qué creés que significa?
- La celda de abajo calcula qué porcentaje de las filas de cada localidad lo tiene. ¿Está repartido
  parejo? ¿Por qué eso importa para la pregunta del TP?

> 💡 **Pista.** Compará ese valor con los consumos de las primeras filas del archivo: ¿cuántas veces
> más grande es? En los porcentajes, buscá la localidad que se despega de las demás, y pensá qué le
> pasa a su promedio si ese valor queda adentro.

In [ ]:
mas_repetido = df["consumo"].value_counts().index[0]
print(f"El consumo que más se repite es {nro(mas_repetido)} kWh: aparece "
      f"{(df['consumo'] == mas_repetido).sum()} veces.\n")
print("% de las filas de cada localidad que lo tienen:")
((df["consumo"] == mas_repetido).groupby(df["localidad"]).mean() * 100).round(1)

*Escribí tu respuesta acá (doble clic para editar esta celda).*

### ✍️ P2 · Los consumos con coma

Hay consumos escritos con coma decimal y muy chicos: menos de 3 kWh en todo un mes, lo que gasta una
heladera en un par de días. La celda de abajo los busca y hace algunas cuentas.

- ¿De qué localidad y de qué meses son? ¿Qué te dice el importe de esas filas sobre lo que pasó?
- Mirá la mediana de Formosa de las tres formas. ¿Qué harías con esas filas? ¿Por qué borrarlas
  también estaría mal?

> 💡 **Pista.** Compará los pesos por kWh de esas filas con los del resto: ¿cuántas veces más da? Si a
> esos hogares se les cobró lo normal, lo que está mal no es el importe, es el consumo: ¿en qué otra
> unidad se mide la energía? En las tres medianas, fijate en qué lugar queda Formosa contra Posadas
> cada vez, y pensá qué tiempo hace en el NEA de octubre a diciembre.

In [ ]:
chicos = df[df["consumo"] < 10]                  # consumos de menos de 10 kWh en un mes
print(f"{len(chicos)} filas con consumos de menos de 10 kWh. De dónde son:")
print(chicos[["localidad", "periodo"]].value_counts().to_string())

pesos_por_kwh = df["importe_pesos"] / df["consumo"]
resto = (df["consumo"] >= 10) & (df["consumo"] != mas_repetido)
print(f"\nPesos facturados por kWh · en esas filas: {nro(pesos_por_kwh[df['consumo'] < 10].median())}"
      f"   ·   en el resto: {nro(pesos_por_kwh[resto].median())}")

formosa = df.loc[(df["localidad"] == "Formosa") & (df["consumo"] != mas_repetido), "consumo"]
print("\nMediana del consumo de Formosa:")
print(f"  dejando esas filas como están:  {nro(formosa.median())} kWh")
print(f"  borrándolas:                    {nro(formosa[formosa >= 10].median())} kWh")
print(f"  multiplicándolas por 1.000:     {nro(formosa.where(formosa >= 10, formosa * 1000).median())} kWh")
posadas = df.loc[(df["localidad"] == "Posadas") & (df["consumo"] != mas_repetido), "consumo"]
print(f"\nY la de Posadas, para comparar: {nro(posadas.median())} kWh")

*Escribí tu respuesta acá (doble clic para editar esta celda).*

### Tu limpieza

Con las filas duplicadas que contaste más arriba y con lo que encontraste en las preguntas 1 y 2,
completá los huecos `___` de la celda de abajo. Son tus decisiones: el resto del código ya está hecho
y las aplica. Ojo, que cada decisión cambia todos los números que vienen después.

In [ ]:
# ─── TUS DECISIONES: completá los ___ ────────────────────────────────────────
SACAR_DUPLICADOS = ___     # ¿sacás las filas duplicadas? True (sí) o False (no)
CODIGO = ___               # el valor de la P1, que no es un consumo de verdad (un número)
LOCALIDAD = ___            # la localidad de la P2, entre comillas: por ejemplo "Corrientes"
DESDE = ___                # el primer mes afectado, entre comillas: por ejemplo "2025-06"
FACTOR = ___               # por cuánto hay que multiplicar esos consumos (un número)
# ─────────────────────────────────────────────────────────────────────────────

if None in (SACAR_DUPLICADOS, CODIGO, LOCALIDAD, DESDE, FACTOR):
    raise NotImplementedError("Faltan completar los huecos ___ de la limpieza.")

datos = df.drop_duplicates() if SACAR_DUPLICADOS else df
datos = datos.copy()
datos.loc[datos["consumo"] == CODIGO, "consumo"] = np.nan                       # no es un consumo
afectadas = (datos["localidad"] == LOCALIDAD) & (datos["periodo"] >= DESDE)
datos.loc[afectadas, "consumo"] = (datos.loc[afectadas, "consumo"] * FACTOR).round()
datos["mes"] = datos["periodo"].str[5:].astype(int)                              # "2025-03" -> 3
print(f"datos: {nro(len(datos))} filas, listas para analizar")

🔍 **Qué controla la celda de abajo.** Compara tu tabla `datos` con los valores que tiene que tener si
la limpieza quedó bien: cuántas filas y cuántos hogares quedan, cuántos consumos quedaron como
faltantes (`NaN`), y que no haya consumos imposibles, ni demasiado chicos ni demasiado grandes. Si algo
da ❌, revisá tus decisiones.

In [ ]:
# 🔒 Checkpoint de la limpieza. Si algo da ❌, revisá tus decisiones antes de seguir.
verificar("filas", len(datos), 9600)
verificar("hogares", datos["id_hogar"].nunique(), 800)
verificar("consumos faltantes", int(datos["consumo"].isna().sum()), 245)
verificar("ningún consumo por debajo de 50 kWh", bool(datos["consumo"].min() >= 50), True)
verificar("ningún consumo por encima de 5.000 kWh", bool(datos["consumo"].max() <= 5000), True)

---
## Parte 2 · ¿Cómo es el consumo por dentro?  *(punto 2)*

Ahora sí, las fórmulas. Las cuatro de esta parte se escriben **a mano**: con `sum()`, `len()`, un
`for` y `math.sqrt()`, **sin pandas ni numpy adentro de la función**. Reciben una **lista de números
sin faltantes**: la notebook se encarga de sacar los `NaN` antes de pasársela.

**Ejercicios 1 y 2.** `media()` y `varianza()`.

In [ ]:
def media(valores):
    """El promedio: la suma de todos los valores dividida por cuántos son.

        media([2, 4, 9])   ->   5.0
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Sumá todos los valores con sum().
    #   2. Dividí esa suma por cuántos valores hay: len(valores).
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar media(). Mirá el TODO justo arriba.")

In [ ]:
def varianza(valores):
    """El promedio de las distancias a la media, elevadas al cuadrado.

    Como los datos son una muestra, la suma se divide por n - 1 (y no por n).

        varianza([2, 4, 9])   ->   13.0
        (la media es 5; las distancias, -3, -1 y 4; al cuadrado, 9 + 1 + 16 = 26; 26 / 2 = 13)
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Calculá la media con tu media(): tenés que usarla.
    #   2. Para cada valor v, calculá (v - la media) ** 2, y sumá todo.
    #   3. Dividí esa suma por len(valores) - 1.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar varianza(). Mirá el TODO justo arriba.")

🔍 **Qué prueba la celda de abajo.** Primero, el ejemplo del docstring, que se puede hacer con lápiz y
papel. Después, tus funciones contra las de pandas, sobre todo el consumo y sobre Isla del Cerrito
sola: `.mean()` es el promedio de pandas y `.var()`, su varianza. `.dropna().tolist()` saca los
faltantes y convierte la columna en una lista, que es lo que reciben tus funciones.

Al final muestra, sin verificar, la varianza que da **numpy** (`np.var()`). Fijate que no es igual a
la tuya: es la pregunta P3.

In [ ]:
# 🔒 Verificación de los ejercicios 1 y 2: contra pandas.

# El ejemplo del docstring.
verificar("media([2, 4, 9])", media([2, 4, 9]), 5.0)
verificar("varianza([2, 4, 9])", varianza([2, 4, 9]), 13.0)

# Todo el consumo y la isla sola: tu función (con una lista) contra pandas (con la columna).
isla = datos.loc[datos["localidad"] == "Isla del Cerrito", "consumo"]
for nombre, columna in [("todo el consumo", datos["consumo"]), ("Isla del Cerrito", isla)]:
    v = columna.dropna().tolist()                 # la lista que reciben tus funciones
    verificar(f"media · {nombre}", media(v), columna.mean())
    verificar(f"varianza · {nombre}", varianza(v), columna.var())

# La varianza de numpy, sólo para mirar (es la P3).
print()
for nombre, columna in [("todo el consumo", datos["consumo"]), ("Isla del Cerrito", isla)]:
    v = columna.dropna().tolist()
    print(f"{nombre:17s} tu varianza: {nro(varianza(v), 1):>11}   ·   la de numpy: {nro(np.var(v), 1):>11}"
          f"   ·   diferencia: {nro((varianza(v) / np.var(v) - 1) * 100, 3)}%   ·   n = {len(v)}")

### ✍️ P3 · Tu varianza y la de numpy

Tu varianza coincide con la de pandas, pero no con la de numpy.

- ¿Por qué? Fijate en el docstring de tu función por cuánto se divide (y averiguá por cuánto divide
  numpy).
- ¿Por qué la diferencia es mucho más grande en Isla del Cerrito que en todo el consumo?

> 💡 **Pista.** En la salida, mirá la `n` de cada renglón. Probá con números redondos: dividir la
> misma suma por 9 en vez de por 10, ¿cuánto cambia el resultado? ¿Y dividirla por 9.999 en vez de por
> 10.000?

*Escribí tu respuesta acá (doble clic para editar esta celda).*

**Ejercicios 3 y 4.** `desvio()` y `coef_variacion()`. Las dos se arman con las que ya escribiste.

In [ ]:
def desvio(valores):
    """El desvío estándar: la raíz cuadrada de la varianza.

    La varianza queda en unidades al cuadrado (kWh²); la raíz la devuelve a kWh.

        desvio([2, 4, 9])   ->   3.6056   (la raíz de 13)
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   Devolvé math.sqrt() de tu varianza(valores). Es una línea, y tenés que
    #   usar tu varianza().
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar desvio(). Mirá el TODO justo arriba.")

In [ ]:
def coef_variacion(valores):
    """El coeficiente de variación: el desvío dividido por la media.

    Es la dispersión RELATIVA, sin unidades: sirve para comparar grupos de tamaños distintos.

        coef_variacion([2, 4, 9])   ->   0.7211   (3,6056 / 5)
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   Dividí tu desvio(valores) por tu media(valores). Tenés que usar las dos.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar coef_variacion(). Mirá el TODO justo arriba.")

🔍 **Qué prueba la celda de abajo.** Lo mismo que antes: el ejemplo del docstring, y tus funciones
contra pandas. El desvío de pandas es `.std()`; el coeficiente de variación no tiene una función
propia, así que se calcula dividiendo `.std()` por `.mean()`.

In [ ]:
# 🔒 Verificación de los ejercicios 3 y 4: contra pandas.
verificar("desvio([2, 4, 9])", desvio([2, 4, 9]), math.sqrt(13))
verificar("coef_variacion([2, 4, 9])", coef_variacion([2, 4, 9]), math.sqrt(13) / 5)

for nombre, columna in [("todo el consumo", datos["consumo"]), ("Isla del Cerrito", isla)]:
    v = columna.dropna().tolist()
    verificar(f"desvío · {nombre}", desvio(v), columna.std())
    verificar(f"coeficiente de variación · {nombre}", coef_variacion(v),
              columna.std() / columna.mean())

### Cada localidad, antes y después de limpiar

Esta celda ya está hecha: usa tus funciones para comparar cada localidad **antes** de limpiar (el
archivo tal cual) y **después**. También muestra cuántos hogares tiene cada una y cuántos aires
acondicionados tienen, en promedio.

In [ ]:
tabla = []
for loc in datos["localidad"].unique():
    antes = df.loc[df["localidad"] == loc, "consumo"].dropna()
    despues = datos.loc[datos["localidad"] == loc, "consumo"].dropna()
    enero = datos[(datos["localidad"] == loc) & (datos["mes"] == 1)]    # una fila por hogar
    tabla.append({
        "localidad": loc,
        "hogares": len(enero),
        "aires por hogar": media(enero["aires_acondicionados"].tolist()),
        "media antes": media(antes.tolist()),
        "mediana antes": antes.median(),
        "media": media(despues.tolist()),
        "mediana": despues.median(),
        "desvío": desvio(despues.tolist()),
        "coef. de variación": coef_variacion(despues.tolist()),
    })
pd.DataFrame(tabla).set_index("localidad").sort_values("mediana", ascending=False).round(2)

### ✍️ P4 · Isla del Cerrito, antes y después

- ¿Cuánto cambiaron la media y la mediana de Isla del Cerrito al limpiar? ¿Cuál se movió más, y por
  qué?
- Si alguien hubiera usado la media de antes, ¿qué titular habría escrito? Escribilo.
- Después de limpiar, Isla del Cerrito es la localidad que menos consume. ¿Cuántos hogares sostienen
  esa afirmación? ¿La publicarías así?

> 💡 **Pista.** En la tabla, compará la fila de Isla del Cerrito con las demás en «media antes» y
> «mediana antes», y mirá la columna «hogares». Volvé a la P1: ¿qué porcentaje de las filas de la isla
> tenía ese código? Pensá cuánto pesa un valor gigante en una suma, y cuánto cuando sólo importa el
> orden.

*Escribí tu respuesta acá (doble clic para editar esta celda).*

---
## Parte 3 · ¿Qué se relaciona con el consumo?  *(punto 3)*

**Ejercicio 5.** La última fórmula a mano: `correlacion()`, el r de Pearson. Mide cuánto se parecen
los puntos de dos variables a una recta: va de −1 a 1.

In [ ]:
def correlacion(x, y):
    """El r de Pearson: la covarianza dividida por el producto de los dos desvíos.

        covarianza = suma de (x - media de x) * (y - media de y), dividida por (n - 1)
        r          = covarianza / (desvío de x * desvío de y)

        correlacion([1, 2, 3], [2, 4, 7])   ->   0.9934

    x e y son dos listas del mismo largo, sin faltantes.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Calculá la media de x y la media de y con tu media().
    #   2. Recorré los pares con  for a, b in zip(x, y):  y sumá
    #      (a - media de x) * (b - media de y).
    #   3. Dividí esa suma por len(x) - 1: es la covarianza.
    #   4. Devolvé la covarianza dividida por desvio(x) * desvio(y).
    #   Tenés que usar tu media() y tu desvio().
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar correlacion(). Mirá el TODO justo arriba.")

🔍 **Qué prueba la celda de abajo.** El ejemplo del docstring (contra numpy) y tu r contra el de
pandas (`.corr()`) para dos pares: temperatura × consumo y aires × consumo. Como tu función no acepta
faltantes, primero se arma `par`: las dos columnas, sin las filas donde falta alguna (`.dropna()`).

In [ ]:
# 🔒 Verificación del ejercicio 5: contra numpy y pandas.
verificar("correlacion([1, 2, 3], [2, 4, 7])", correlacion([1, 2, 3], [2, 4, 7]),
          np.corrcoef([1, 2, 3], [2, 4, 7])[0, 1])

for col in ("temperatura_media", "aires_acondicionados"):
    par = datos[[col, "consumo"]].dropna()                # sin las filas a las que les falta algo
    verificar(f"r({col}, consumo)", correlacion(par[col].tolist(), par["consumo"].tolist()),
              par[col].corr(par["consumo"]))

### La temperatura

Con tu `correlacion()`, temperatura × consumo da un r chico: está en el título del gráfico de abajo.
La lectura apurada es *"la temperatura casi no influye en cuánta luz se gasta"*. Antes de creerla,
mirá el gráfico (ya está hecho): los puntos grises son 3.000 lecturas al azar, y los rojos, la mediana
del consumo para cada temperatura.

In [ ]:
muestra = datos.dropna(subset=["consumo"]).sample(3000, random_state=42)
medianas = datos.groupby("temperatura_media")["consumo"].median()
par = datos[["temperatura_media", "consumo"]].dropna()
r = correlacion(par["temperatura_media"].tolist(), par["consumo"].tolist())

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.scatterplot(data=muestra, x="temperatura_media", y="consumo", s=12, alpha=0.25, color=GRIS,
                edgecolor=None, label="3.000 lecturas al azar", ax=ax)
ax.plot(medianas.index, medianas.values, "o", color=ROJO, label="mediana de cada temperatura")
ax.set_title(f"Temperatura × consumo   ·   r = {nro(r, 2)}")
ax.set_xlabel("temperatura media del mes (°C)")
ax.set_ylabel("consumo (kWh por mes)")
ax.legend()
plt.show()

### ✍️ P5 · La temperatura

- ¿Qué forma tiene la relación? ¿Por qué tiene sentido en el NEA?
- ¿Por qué el r da tan bajo si la relación es tan clara?

> 💡 **Pista.** Seguí los puntos rojos de izquierda a derecha: ¿dónde está el consumo más bajo, y qué
> pasa hacia cada costado? Pensá qué se enchufa en una casa cuando hace calor y cuando hace frío. Para
> el r, imaginá la recta que mejor pasa por esos puntos rojos: ¿qué inclinación tendría?

*Escribí tu respuesta acá (doble clic para editar esta celda).*

### Nivel de subsidio y aires acondicionados

Para cruzar características **de los hogares** hay que contar hogares, no filas: si no, cada hogar
pesa doce veces. Por eso `una_por_hogar` se queda con una sola fila de cada hogar (la de enero).

La celda muestra la mediana del consumo de cada nivel, el r de aires × consumo (con tu
`correlacion()`) y una tabla con el porcentaje de los hogares de cada nivel que tiene 0, 1, 2, 3 o 4
aires: cada fila suma 100.

In [ ]:
una_por_hogar = datos[datos["mes"] == 1]                  # una fila por hogar: la de enero
print("Mediana del consumo de cada nivel:",
      {nivel: nro(m) for nivel, m in datos.groupby("nivel_subsidio")["consumo"].median().items()})
par = datos[["aires_acondicionados", "consumo"]].dropna()
print("r de aires × consumo:",
      nro(correlacion(par["aires_acondicionados"].tolist(), par["consumo"].tolist()), 2))

pd.crosstab(una_por_hogar["nivel_subsidio"], una_por_hogar["aires_acondicionados"],
            normalize="index").mul(100).round(1)

### ✍️ P6 · ¿Los de más ingresos consumen más?

Los hogares N1 consumen bastante más que los N2. La explicación obvia es el ingreso.

- Con la tabla de arriba: ¿qué otra cosa puede estar detrás de esa diferencia?
- Releé el diccionario de datos: ¿en qué orden van N1, N2 y N3 si los ordenás por ingreso? ¿En qué
  gráfico de este TP te importa ese orden?

> 💡 **Pista.** Cada fila de la tabla suma 100: compará N1 y N2 en las columnas de 3 y 4 aires, y en la
> de 0. Mirá también el r de aires × consumo. Y pensá qué es lo que gasta luz en una casa: ¿el ingreso,
> o lo que está enchufado?

*Escribí tu respuesta acá (doble clic para editar esta celda).*

---
## Parte 4 · El dashboard  *(punto 4)*

El cierre del TP es **una figura de 2 × 2** que conteste la pregunta del TP y se entienda sin abrir la
notebook. Esta la armás vos.

Primero, un ejemplo de cómo se arma una figura con varios gráficos. No es el dashboard: es para ver
la mecánica.

In [ ]:
# Un ejemplo de subplots: dos gráficos, uno al lado del otro.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))      # 1 fila y 2 columnas de gráficos

# Cada gráfico de seaborn recibe ax=..., que dice en cuál de los cuadros se dibuja.
q1, q3 = una_por_hogar["superficie_m2"].quantile([0.25, 0.75])
sns.histplot(data=una_por_hogar, x="superficie_m2", bins=30, ax=axes[0])
axes[0].axvline(q1, color=ROJO, ls="--")             # una línea vertical en x = q1
axes[0].axvline(q3, color=ROJO, ls="--")
axes[0].set_title(f"La mitad de las viviendas tiene entre {nro(q1)} y {nro(q3)} m²")
axes[0].set_xlabel("superficie (m²)")
axes[0].set_ylabel("hogares")

moda = una_por_hogar["personas"].mode()[0]
sns.countplot(data=una_por_hogar, x="personas", color=AZUL, ax=axes[1])
axes[1].set_title(f"Lo más común son los hogares de {moda} personas")
axes[1].set_xlabel("personas en el hogar")
axes[1].set_ylabel("hogares")

fig.suptitle("Cómo son los hogares del NEA", fontsize=14)    # el título de toda la figura
plt.tight_layout()
plt.show()

### Tu dashboard

Cada cuadrante contesta una parte de la pregunta del TP. La tabla dice qué tiene que mostrar cada uno
y qué te puede servir; **las decisiones son tuyas**: qué mostrar exactamente, los títulos, el orden y
los colores.

| cuadrante | qué tiene que mostrar | te puede servir |
|---|---|---|
| `axes[0, 0]` | la forma del consumo limpio, con la media y la mediana | `sns.histplot()` y `ax.axvline()` |
| `axes[0, 1]` | el consumo de cada localidad | `sns.boxplot()`, con `order=`, `hue=` y `palette=COLORES` |
| `axes[1, 0]` | temperatura × consumo, que se vea la forma de la P5 | `sns.scatterplot()` con una muestra y `alpha=`, o las medianas por temperatura |
| `axes[1, 1]` | nivel de subsidio × aires acondicionados | `pd.crosstab()` y `.plot(kind="barh", stacked=True, ax=...)`, o `sns.heatmap()` |

**Lo que se evalúa:**

- cada título dice **lo que se ve**, no el nombre de la variable ("Formosa consume más", no "consumo
  por localidad");
- los ejes tienen nombre y **unidad** (kWh por mes, °C, % de hogares);
- el **n** está a la vista donde importa: por ejemplo, cuántos hogares tiene cada localidad;
- cada localidad tiene **siempre el mismo color** (`COLORES`);
- ningún eje está **truncado en silencio**;
- el título de la figura (`fig.suptitle`) es **el hallazgo**, en una frase;
- queda **a la vista en la salida de la celda** (es lo que se evalúa).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# [0, 0] La forma del consumo


# [0, 1] El consumo de cada localidad


# [1, 0] Temperatura × consumo


# [1, 1] Nivel de subsidio × aires acondicionados


fig.suptitle("___", fontsize=16)
plt.tight_layout()
plt.show()

### ✍️ El hallazgo, en tres líneas

La respuesta a la pregunta del TP, para acompañar tu dashboard.

| línea | qué tiene que tener |
|---|---|
| 1 | qué se ve, para alguien que no sabe estadística |
| 2 | el número **y** el `n` |
| 3 | una razón **concreta** para dudar: qué no podés afirmar todavía |

La línea 3 es donde falla casi todo el mundo: se escribe una vaguedad (*"habría que investigar
más"*) en vez de una razón concreta que salga de lo que encontraste en este TP.

> 💡 **Pista.** Los números de la línea 2 están en la tabla de cada localidad (Parte 2). Para la
> línea 3, repasá tus respuestas: ¿qué decisión de la limpieza cambia qué localidad sale primera? Y lo
> que viste en la P6 con los aires, ¿puede pasar también entre localidades? Mirá la columna «aires por
> hogar» de esa misma tabla.

*Escribí tu respuesta acá (doble clic para editar esta celda).*

---
## 🔒 Control de reglas

Revisa que tus cinco funciones no usen pandas ni numpy. Corré esta celda antes de entregar.

🔍 **Cómo lo hace.** `inspect.getsource()` le pide a Python el código de cada función, tal como lo
escribiste. La celda le saca el docstring y los comentarios, y en lo que queda busca palabras
prohibidas, como `np.`, `pandas` o `.mean(`. Para cada función muestra ✅ si está todo bien, ❌ y lo que
encontró si usa algo prohibido, o ⏳ si todavía no la completaste.

In [ ]:
# 🔒 Control de reglas. No modifiques esta celda.
PROHIBIDO = ["np.", "numpy", "pd.", "pandas", "statistics", ".mean(", ".var(", ".std(", ".corr("]

for f in [media, varianza, desvio, coef_variacion, correlacion]:
    codigo = inspect.getsource(f)                     # el código de la función, como texto
    # Lo que está después del docstring, sin los renglones de comentario.
    cuerpo = codigo.split('"""')[-1] if codigo.count('"""') >= 2 else codigo
    cuerpo = "\n".join(l for l in cuerpo.splitlines() if not l.strip().startswith("#"))
    usados = [p for p in PROHIBIDO if p in cuerpo]
    if "NotImplementedError" in cuerpo:
        print(f"⏳ {f.__name__}: todavía sin completar")
    elif usados:
        print(f"❌ {f.__name__} usa {', '.join(usados)}")
    else:
        print(f"✅ {f.__name__}")

### Antes de entregar

- [ ] *Entorno de ejecución → Reiniciar sesión y ejecutar todo*: la notebook corre entera, sin errores.
- [ ] Las cinco funciones dan ✅, y el control de reglas también.
- [ ] No quedó ningún hueco `___` sin completar.
- [ ] Las seis respuestas ✍️ y el hallazgo están escritos, y cada uno cita un número o un gráfico de
  tu notebook.
- [ ] El dashboard se ve en la salida de la celda.
- [ ] Descargaste la notebook con *Archivo → Descargar → Descargar .ipynb*. **Ese único archivo es lo que se
  entrega** (sin `dashboard.png`, capturas ni `.csv`).